# Subsample a set of input genomes to minimize overrepresentation in gene-by-gene schema

## Introduction

### Aim
Subsample a set of Giardia genomes to reduce overrepresentation of highly similar assemblies in a gene-by-gene schema


### Analysis steps

1. Project a distance matrix obtained using **mash** (kmer size = 51) to a vector space (**t-SNE**)
2. Use a clustering method (**HDBScan**) to determine the similarity among them. 
3. After selecting (through visualization) the ideal parameters for clustering in the given dataset, conduct a grouped subsample.

The **data input** is a square matrix of pre-calculated mash distances (**sourmash v4.8.9**) from n genome assemblies determined to be of good enough quality based on assembly metrics.

## Define dependencies and load data

The python environment required for this notebook can be reproduced using the `environment.yml` on this directory.

In [ ]:
    # Import all dependencies 

import os
import hdbscan
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as colormaps
from joblib import Parallel, delayed
from sklearn.manifold import TSNE
from sklearn.cluster import HDBSCAN
from imblearn.under_sampling import RandomUnderSampler
from IPython.display import display, HTML


Load and sanitize the data, define a seed for reproducibility, and calculate the sample size (number of inputs) in the dataset

In [ ]:
sour_data = pd.read_csv('../../output/sourmash/sourmash_distances_hq_genomes.csv', sep=',')
sour_metadata = pd.read_csv('../../input_data/metadata/merged_SraPrystajecky.csv', sep=',')

# Clean missing metadata values
sour_metadata = sour_metadata.replace(['nan', 'NaN', 'missing', 'NA', None], '')
sour_metadata = sour_metadata.fillna('')

# Sanitize the labels
sour_data.columns = sour_data.columns.str.replace('.*/', '', regex=True)
sour_data.index = sour_data.columns

# Random seed for reproducibility, set sample size value
seed = 112233
sample_size = sour_data.shape[1]


In [ ]:
# Lookup dictionaries to connect metadata and distance matrix
isolate_map     = {}
host_map        = {}
country_map     = {}
location_map    = {}

for _, row in sour_metadata.iterrows():
    # Get best possible location data
    country = str(row['country_geoloc']).lower().strip()
    region = str(row['region_geoloc']).lower().strip()
    if country and region:
        location_value = f"{country}_{region}"
    else:
        location_value = country if country else (region if region else "Unknown")

    for identifier in [row['run'], row['biosample']]:
        isolate_map[identifier]     = str(row['isolate']).replace("/", "-")
        host_map[identifier]        = str(row['isolation_source'])
        country_map[identifier]     = country
        location_map[identifier]    = location_value


# Function to label data (handles SRX_SRR and SRR formats)
def get_custom_label(raw_id):
    key = raw_id.split("_")[-1] if "_" in raw_id else raw_id

    if key in isolate_map:
        return(
            f"{key}_"
            f"{isolate_map[key]}_"
            f"{host_map[key]}_"
            f"{location_map[key]}"
        )
    
    return raw_id

## t-SNE projection

### Hyperparameter tuning [~ 30 seconds with 8 cores]

This loop will show us different topologies according to the perplexity value (influence of neighbors in clustering). The metric is set to `precomputed` as it comes directly from a MASH square distance matrix. 

I am testing perplexity values below 50, as recommended in the t-SNE method, and evaluating between 2500 and 5000 iterations for convergence. 


In [ ]:
# ---------------------------------------------------------------------------------------------------
# MANUAL INPUT: define values for perplexity (recommendation: max of 50)
perplexity = np.arange(10, 51, 10)
# ---------------------------------------------------------------------------------------------------


# Set iterations
iters =  (1000, 2500, 5000)
cols = math.ceil(len(perplexity) / 3)

# Define a function to run every tsne embedding
def run_single_tsne(perplexity_val, iterations, data, seed):
    model = TSNE(
        n_components=2,
        metric='precomputed',
        init='random',
        perplexity=perplexity_val,
        max_iter=iterations, 
        random_state=seed
    )

    fitted = model.fit_transform(data)
    return fitted, model.kl_divergence_


# Capture divergence value for each iteration
for value in iters:
            
    fig, axs = plt.subplots(3, cols, figsize=(12,15),
                            facecolor='w', layout="constrained")
    fig.suptitle(
        f"Topology at different perplexity (p) values with {value} iterations",  
        fontsize=16, va='bottom', weight='bold'
        ) 
    axs = axs.ravel()

    # Prepare parallelized execution
    results = Parallel(n_jobs=-1)(
        delayed(run_single_tsne)(p, value, sour_data, seed) for p in perplexity
    )
    divergence = []

    # Fit [KL divergence] with perplexity values
    for index, (p,res) in enumerate(zip(perplexity, results)):

        fitted, kl_divergence = res
        divergence.append(kl_divergence)
        
        # Plot 2D projections
        axs[index].scatter(fitted[:, 0], fitted[:,1], color=plt.cm.tab20(index) )
        axs[index].set_title(f"p = {p}", fontsize=14)
        
    # Last subplot with perplexity vs KL divergence 
    axs[5].plot(perplexity, divergence, color='red',
                marker='o', fillstyle='full')
    axs[5].set_title("Perplexity vs KL Divergence", fontsize=14)
    axs[5].set_xlabel("Perplexity")

    plt.show()

**WARNING:** The choice of hyperparameter may vary when you run it, and it is inherent to each dataset.  
**RECOMMENDATION:** Visualize your data to choose the optimal hyperparameters.

### Optional: verify topology with chosen parameter [ ~1 min with 8 cores]

We find convergence of clustering with at least 2500 iterations using a perplexity value around 30. The input dataset includes the assemblies passing the QUAST filter. 
- Here we run the model with the chosen perplexity value to verify if the same topology is reproduced (**see below**) across multiple instances.

In [ ]:
%%script false --no-raise-error
# ---------------------------------------------------------------------------------------------------
# MANUAL INPUT: chosen perplexity value
chosen_perplexity = 30
# ---------------------------------------------------------------------------------------------------

fig, axs = plt.subplots(3, 3, figsize=(15,15), 
                facecolor='w', layout="constrained")
fig.suptitle("Examples of iterations with selected hyperparameters",
        fontsize=16, va='bottom', weight='bold') 
axs = axs.ravel()

results = Parallel(n_jobs=-1)(
        delayed(run_single_tsne)(chosen_perplexity, 2500, sour_data, seed) for _ in range(0,9)
        )


for index, res in enumerate(results):
    fitted = res[0]
    axs[index].scatter(fitted[:, 0], fitted[:, 1],
                        color=plt.cm.tab20(index))
    axs[index].set_title(f"Run {index + 1}", fontsize=16)

plt.show()

### Save the final embedding coordinates [~1 min with 8 cores]

The embeddings from a fitted model (with a random seed for reproducibility) are saved. 
- Proper labels are attached to this new data frame based on the input data.

In [ ]:
# ---------------------------------------------------------------------------------------------------
# MANUAL INPUT: chosen perplexity value
chosen_perplexity = 30
# ---------------------------------------------------------------------------------------------------

# Save embeddings into a pandas dataframe
tsne, _ = run_single_tsne(chosen_perplexity, 2500, sour_data, seed)
vector_mat = pd.DataFrame(tsne, columns=['t-SNE-1', 't-SNE-2'])
vector_mat.index= sour_data.index

# Add relevant metadata to coordinates
vector_mat['raw_id'] = vector_mat.index.to_series().str.split('_').str[-1]
vector_mat['Host'] = vector_mat['raw_id'].map(host_map).fillna("Unknown")
vector_mat['Location'] = vector_mat['raw_id'].map(location_map).fillna("Unknown")
vector_mat['Country'] = vector_mat['raw_id'].map(country_map).fillna("Unknown")

In [ ]:
import seaborn as sns

plt.figure(figsize=(12,8))
sns.set_style("ticks")
plot = sns.scatterplot(
    data=vector_mat, 
    x='t-SNE-1', 
    y='t-SNE-2', 
    hue='Country', 
    style='Host', # You can use a second metadata layer as the point shape
    s=100,            # Marker size
    alpha=0.7,
    palette='tab20'
)


## HDBSCAN clustering

### Define a ploting function

The function takes a numpy array with the clustering labels to produce a scatter plot colored by cluster [_if available_]. 

- The size of a point represents the probability of belonging to its cluster
- Unclustered values are marked with a black "X"

In [ ]:
def plot(X, labels, probabilities=None, parameters=None, ground_truth=False, ax=None):
    
    # optional: transform pd.dataframe to numpy
    if isinstance(X, pd.DataFrame):
        X = X.values  
    if isinstance(labels, pd.Series):
        labels = labels.values
    
    # creates a new ploting space if axes is not specified
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 7))
    
    # labels and probabilities are set to "1" if not specified
    labels = labels if labels is not None else np.ones(X.shape[0])
    probs = probabilities if probabilities is not None else np.ones(X.shape[0])
    
    # selects colors in spectra palette according to set number
    unique_labels = set(labels)
    colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]
    
    # The probability of a point belonging to its labeled cluster determines
        # the size of its marker
    proba_map = {idx: probs[idx] for idx in range(len(labels))}    
    for k, col in zip(unique_labels, colors):
        if k == -1:
            # Black used for noise.
            col = [0, 0, 0, 1]
        class_index = np.where(labels == k)[0]
        for ci in class_index:
            ax.plot(
                X[ci, 0],
                X[ci, 1],
                "x" if k == -1 else "o",
                markerfacecolor=tuple(col),
                markeredgecolor="k",
                markersize=6 if k == -1 else 5 + 7 * proba_map[ci],
            )
    
    # Improve labelling
    n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
    preamble = "True" if ground_truth else "Estimated"
    title = f"{preamble} number of clusters: {n_clusters_}"
    if parameters is not None:
        parameters_str = ", ".join(f"{k}={v}" for k, v in parameters.items())
        title += f" \n {parameters_str}"
    ax.set_title(title,  fontsize=14)
    plt.tight_layout()

### Hyperparameter tuning [**~ 30 sec each**]

**WARNING:** your choice of hyperparameter may vary according to your input, visualize to decide. 

HDBSCAN will find the optimal epsilon automatically, but we should tune the `min_cluster_size` and the `min_samples` values. 

- We see that small `min_cluster_sizes` (~ 3) are not ideal as they create many small clusters with doubtful separation
- If we go above 30, it fails to select one region with obvious high density
- We want to be somehow conservative and only samples with a definitive association so better to go with an intermediate value 0f 12

In [ ]:
# ---------------------------------------------------------------------------------------------------
# MANUAL INPUT: set of sensible cluster sizes
cluster_sizes = [3, 5, 7, 10, 12, 15, 20, 25, 30]
# ---------------------------------------------------------------------------------------------------


fig, axes = plt.subplots(3, 3, figsize=(12,18))
axes = axes.ravel()

for i, size in enumerate(cluster_sizes):
    # pass cluster size into function
    hdb = HDBSCAN(
        algorithm="auto", 
        cluster_selection_method="eom", 
        min_cluster_size=size,
        copy=True
    ).fit(vector_mat)
    
    # create a dictionary for plotting
    param_dict = {"min_cluster_size": size}
    plot(vector_mat, hdb.labels_, hdb.probabilities_, param_dict, ax=axes[i])

The larger the value of `min_samples` you provide, the more conservative the clustering. We will explore the ideal case for our dataset, using the `min_cluster_size` selected in the previous step.

In [ ]:
# ---------------------------------------------------------------------------------------------------
# MANUAL INPUT: input chosen cluster size and values to explore for min_samples
selected_min_cluster_size = 12
min_samples = np.arange(3, 30, 3)
# ---------------------------------------------------------------------------------------------------


fig, axes = plt.subplots(3, 3, figsize=(12,18))
axes = axes.ravel()

for i, val in enumerate(min_samples):
    hdb_fit = HDBSCAN(
        algorithm="auto", 
        cluster_selection_method="eom",
        min_cluster_size = selected_min_cluster_size,
        min_samples = val,
        copy=True
    ).fit(vector_mat)

    labels = hdb_fit.labels_

    # create a dictionary for plotting
    param_dict = {"min_cluster_size": selected_min_cluster_size,
                  "min_samples": val}
    
    plot(vector_mat, hdb_fit.labels_, hdb_fit.probabilities_, param_dict, ax=axes[i])
    


### Verify (optional) and save final results of HDBScan

**WARNING:** your choice of hyperparameters may vary when you run it. 

We'll go with a `min_cluster_size=12` and `min_samples=10`. 

- The aim is to avoid losing a significant portion of the results 
- We use the implementation in the `hdbscan` module instead of in `sklearn-env` to show the tree-form topology

In [ ]:
hdb_final = hdbscan.HDBSCAN(algorithm="best", cluster_selection_method="eom",
                            min_cluster_size=12, min_samples=12)

hdb_final_fit = hdb_final.fit(vector_mat)

    # plot the selected clustering and the hierarchical tree
fig, axs = plt.subplots(2, 1, figsize=(7, 12))
plot(vector_mat, 
     hdb_final_fit.labels_, 
     hdb_final_fit.probabilities_, 
     ax=axs[0])

hdb_final_fit.condensed_tree_.plot(select_clusters=True)

## Subsampling informed by clustering

A new dataset with labels containing cluster assignment is created for subsampling. To allow visualization of the clustering hierarchy is better to use the library `hdbscan` for the algorithm instead of the `sklearn` implementation

- Non-clustered samples must be retained so they are extracted from the primary data frame
- To balance the overrepresentation of certain clusters we perform undersampling with the `RandomUnderSampler` method from the module `imbalance-learn`

In [ ]:
vector_mat['hdbscan'] = hdb_final_fit.labels_

# sample without replacement, seed for reproducibility
rus=RandomUnderSampler(
    random_state=seed, 
    replacement=False,
    sampling_strategy='auto')

# input whole dataframe and labels
undersampled, labels = rus.fit_resample(vector_mat, vector_mat['hdbscan'])

### Validation and export

Calculate frequency tables showcasing the initial and adjusted distribution of labels.

The data is exported to define the gene-by-gene schema in the `nf_chewbbaca` pipeline

In [ ]:
# Function to create HTML saummary tables
def display_cluster_stats(df, title):
    # Replace '-1' label with unclustered
    clusters = df['hdbscan'].replace(-1, "unclustered")
    
    # Calculate counts and frequencies, merge into a DataFrame
    counts = clusters.value_counts(dropna=False)
    freqs = clusters.value_counts(normalize=True, dropna=False)
    stats = pd.concat([counts, freqs], axis=1, keys=['Count', 'Frequency']).reset_index()
    stats.columns = ['HDBSCAN cluster', 'Count', 'Frequency']
    
    # Format decimals and get total count
    stats['Frequency'] = stats['Frequency'].apply(lambda x: f"{x:.4f}")
    total_count = stats['Count'].sum()
    
    # Render the title and the DataFrame as native HTML
    display(HTML(f"<h3>{title} (Total: {total_count:,})</h3>"))
    display(stats)

# Call the function for both of your matrices
display_cluster_stats(vector_mat, "Initial matrix")
display_cluster_stats(undersampled, "Sampled matrix")


# save only rownames
undersampled.to_csv(
    '../../processed_data/accessions/clustered_subsample_hbscan.txt', 
    sep='\t', 
    header=False, 
    columns=[])
